In [ ]:
import sqlite3

import pandas as pd

from ecommerce_ingestion.config.constants import DB_OUTPUT_BRONZE, DB_OUTPUT_SILVER


In [ ]:
conn = sqlite3.connect(DB_OUTPUT_BRONZE / "oxylabs_sandbox_scraper.db")
data_products = pd.read_sql("SELECT * FROM game_products", conn)
data_categories = pd.read_sql("SELECT * FROM category_nodes", conn)
data_product_categories = pd.read_sql("SELECT * FROM game_product_categories", conn)
data_genre_links = pd.read_sql("SELECT * FROM game_genre_game_link", conn)
data_genre = pd.read_sql("SELECT * FROM game_genre", conn)

data_list = [("data_products", data_products), 
             ("data_categories", data_categories), 
             ("data_product_categories", data_product_categories), 
             ("data_genre_links", data_genre_links),
             ("data_genre", data_genre)]

for name, data in data_list:
    print(f"{name} shape: {data.shape}, {data.shape[1]} columns")

data_products shape: (2976, 11), 11 columns
data_categories shape: (25, 11), 11 columns
data_product_categories shape: (2976, 3), 3 columns
data_genre_links shape: (10004, 2), 2 columns
data_genre shape: (158, 1), 1 columns


In [3]:
data_products.isna().sum()

game_id                       0
game_source_site              0
source_game_product_code      0
game_name                     0
game_product_type           696
game_rating                 454
game_pdp_url                  0
game_developer                4
game_created_at               0
game_updated_at               0
game_description              0
dtype: int64

In [4]:
data_genre_links_grouped = data_genre_links.groupby("game_id").agg(", ".join)
data_genre_links_grouped.shape

(2976, 1)

In [5]:
data_products = data_products.merge(data_product_categories, 
                                    left_on="game_id", 
                                    right_on="game_product_id",
                                    how="left")

data_products = data_products.merge(data_categories, 
                                    left_on="category_id", 
                                    right_on="category_id", 
                                    how="left")

data_products = data_products.merge(data_genre_links_grouped,
                                    left_on="game_id",
                                    right_index=True,
                                    how="left")

data_products.dtypes

game_id                       str
game_source_site              str
source_game_product_code      str
game_name                     str
game_product_type             str
game_rating                   str
game_pdp_url                  str
game_developer                str
game_created_at               str
game_updated_at               str
game_description              str
game_product_id               str
category_id                   str
created_at                    str
category_source_site          str
source_category_code          str
category_name                 str
category_url                  str
category_path                 str
category_parent_id            str
category_level              int64
category_is_leaf            int64
category_created_at           str
category_updated_at           str
genre_id                      str
dtype: object

In [6]:
data_products.isna().sum()

game_id                        0
game_source_site               0
source_game_product_code       0
game_name                      0
game_product_type            696
game_rating                  454
game_pdp_url                   0
game_developer                 4
game_created_at                0
game_updated_at                0
game_description               0
game_product_id                0
category_id                    0
created_at                     0
category_source_site           0
source_category_code           0
category_name                  0
category_url                   0
category_path                  0
category_parent_id          1125
category_level                 0
category_is_leaf               0
category_created_at            0
category_updated_at            0
genre_id                       0
dtype: int64

In [7]:
data_products[["category_id","category_level","category_is_leaf"]].loc[data_products["category_parent_id"].isnull()].nunique()

category_id         5
category_level      1
category_is_leaf    2
dtype: int64

In [8]:
data_products.fillna({"game_product_type": "unknown"}, inplace=True)
data_products.fillna({"game_rating": "unknown"}, inplace=True)
data_products.fillna({"game_developer": "unknown"}, inplace=True)
data_products.fillna({"category_parent_id": "none"}, inplace=True)
data_products.isna().sum()

game_id                     0
game_source_site            0
source_game_product_code    0
game_name                   0
game_product_type           0
game_rating                 0
game_pdp_url                0
game_developer              0
game_created_at             0
game_updated_at             0
game_description            0
game_product_id             0
category_id                 0
created_at                  0
category_source_site        0
source_category_code        0
category_name               0
category_url                0
category_path               0
category_parent_id          0
category_level              0
category_is_leaf            0
category_created_at         0
category_updated_at         0
genre_id                    0
dtype: int64

## Delete Metadata

In [9]:
data_filtered = data_products.drop(axis=1, 
                                   labels=["category_created_at", 
                                           "game_created_at", 
                                           "category_updated_at", 
                                           "game_updated_at",
                                           "created_at"])
data_filtered.head()

,game_id,game_source_site,source_game_product_code,game_name,game_product_type,game_rating,game_pdp_url,game_developer,game_description,game_product_id,category_id,category_source_site,source_category_code,category_name,category_url,category_path,category_parent_id,category_level,category_is_leaf,genre_id
0,The Legend of Zelda: Ocarina of Time,oxylabs_sandbox,1,The Legend of Zelda: Ocarina of Time,singleplayer,E,https://www.metacritic.com/game/nintendo-64/th...,Nintendo,"As a young boy, Link is tricked by Ganondorf, ...",The Legend of Zelda: Ocarina of Time,nintendo/nintendo-64,oxylabs_sandbox,nintendo/nintendo-64,nintendo-64,https://oxylabs.io/products/category/nintendo/...,nintendo/nintendo-64,nintendo,2,1,"action adventure, fantasy"
1,Super Mario Galaxy,oxylabs_sandbox,2,Super Mario Galaxy,singleplayer,E,https://www.metacritic.com/game/wii/super-mari...,Nintendo,[Metacritic's 2007 Wii Game of the Year] The u...,Super Mario Galaxy,nintendo/wii,oxylabs_sandbox,nintendo/wii,wii,https://oxylabs.io/products/category/nintendo/wii,nintendo/wii,nintendo,2,1,"action, platformer, 3d"
2,Super Mario Galaxy 2,oxylabs_sandbox,3,Super Mario Galaxy 2,singleplayer,E,https://www.metacritic.com/game/wii/super-mari...,Nintendo EAD Tokyo,"Super Mario Galaxy 2, the sequel to the galaxy...",Super Mario Galaxy 2,nintendo/wii,oxylabs_sandbox,nintendo/wii,wii,https://oxylabs.io/products/category/nintendo/wii,nintendo/wii,nintendo,2,1,"action, platformer, 3d"
3,Metroid Prime,oxylabs_sandbox,4,Metroid Prime,singleplayer,T,https://www.metacritic.com/game/gamecube/metro...,Retro Studios,Samus returns in a new mission to unravel the ...,Metroid Prime,nintendo/gamecube,oxylabs_sandbox,nintendo/gamecube,gamecube,https://oxylabs.io/products/category/nintendo/...,nintendo/gamecube,nintendo,2,1,"action, shooter, first-person, sci-fi"
4,Super Mario Odyssey,oxylabs_sandbox,5,Super Mario Odyssey,singleplayer,E10+,https://www.metacritic.com/game/switch/super-m...,Nintendo,New Evolution of Mario Sandbox-Style Gameplay....,Super Mario Odyssey,nintendo/switch,oxylabs_sandbox,nintendo/switch,switch,https://oxylabs.io/products/category/nintendo/...,nintendo/switch,nintendo,2,1,"action, platformer, 3d"


In [10]:
data_filtered["game_developer"] = data_filtered["game_developer"].str.lower()

In [11]:
print(data_filtered["game_product_type"].unique())
print(data_filtered["game_rating"].unique())
print(data_filtered["game_developer"].nunique())

<ArrowStringArray>
['singleplayer', 'multiplayer', 'unknown']
Length: 3, dtype: str
<ArrowStringArray>
['E', 'T', 'E10+', 'M', 'unknown', 'K-A', 'RP']
Length: 7, dtype: str
1355


In [ ]:
data_filtered.to_parquet(DB_OUTPUT_SILVER / "silver_data.parquet")
data_genre_links.to_parquet(DB_OUTPUT_SILVER / "game_genre_relationships.parquet")
data_categories.to_parquet(DB_OUTPUT_SILVER / "categories.parquet")